In [1]:
# https://docs.pytorch.org/tutorials/intermediate/char_rnn_generation_tutorial.html

## Learnings
- In RNNs, the network produces `output` and `hidden` for each letter.
- For training,
  - We use the `output` for loss calculation and then it's discarded
  - We use the next `input` letter for the next iteration
  - We feedback the new `hidden` into the next iteration into `hidden`
- In inference,
  - We take the `output`, convert it into its `letter`, then create `letter embedding` and then feed it into the network again until we get `EOS`
  - `hidden` gets initialized and keeps get fed to the network in the loop until we hit `EOS`
- The training example looks like this:
  - for `Nikhit`, the input will be fed one-by-one, which is the embeddings for each letter
  - Then you can give a static `category` for each letter like `Hindi`'s one hot embedding
  - For loss calculation, the `target` will be `ikhit<EOS>` but not the one-hot embedding, it will be the `index` of each letter so that we can calculate NLLoss, if that makes sense
  - for example, for `N`
    - `embedding["N"] => Index["i"]`
    - `embedding["i"] => Index["k"]`
    - `embedding["k"] => Index["h"]`
    - `embedding["h"] => Index["i"]`
    - `embedding["i"] => Index["t"]`
    - `embedding["t"] => Index["<EOS"]`

In [2]:
# Mount the drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
from io import open
import glob
import os
import unicodedata
import string

all_letters = string.ascii_letters + " .,;'-"
n_letters = len(all_letters) + 1 # Plus EOS marker

def findFiles(path): return glob.glob(path)

# Turn a Unicode string to plain ASCII, thanks to https://stackoverflow.com/a/518232/2809427
def unicodeToAscii(s):
    return ''.join(
        c for c in unicodedata.normalize('NFD', s)
        if unicodedata.category(c) != 'Mn'
        and c in all_letters
    )

# Read a file and split into lines
def readLines(filename):
    with open(filename, encoding='utf-8') as some_file:
        return [unicodeToAscii(line.strip()) for line in some_file]

# Build the category_lines dictionary, a list of lines per category
category_lines = {}
all_categories = []
path = "/content/drive/MyDrive/Colab Notebooks/PyTorch Learning/nlp_1_data/names/*.txt"
for filename in findFiles(path):
    category = os.path.splitext(os.path.basename(filename))[0]
    all_categories.append(category)
    lines = readLines(filename)
    category_lines[category] = lines

n_categories = len(all_categories)

if n_categories == 0:
    raise RuntimeError('Data not found. Make sure that you downloaded data '
        'from https://download.pytorch.org/tutorial/data.zip and extract it to '
        'the current directory.')

print('# categories:', n_categories, all_categories)
print(unicodeToAscii("O'Néàl"))

# categories: 18 ['Irish', 'French', 'Russian', 'Polish', 'Greek', 'Czech', 'English', 'Italian', 'Arabic', 'Korean', 'Scottish', 'Japanese', 'German', 'Portuguese', 'Dutch', 'Chinese', 'Spanish', 'Vietnamese']
O'Neal


In [5]:
## Create Training Data

In [6]:
import random

# Random item from a list
def randomChoice(l):
    return l[random.randint(0, len(l) - 1)]

def randomTrainingPair():
  category = randomChoice(all_categories)
  line = randomChoice(category_lines[category])
  return category, line

In [7]:
## Create tensors

In [8]:
def categoryTensor(category):
  cat_tensor = torch.zeros(1, len(all_categories))
  cat_tensor[0][all_categories.index(category)] = 1
  return cat_tensor

def inputTensor(line):
  inp_tensor = torch.zeros(len(line), 1, n_letters)
  for i, letter in enumerate(line):
    inp_tensor[i][0][all_letters.find(letter)] = 1
  return inp_tensor

def targetTensor(line):
  letter_indexes = [all_letters.find(line[li]) for li in range(1, len(line))]
  letter_indexes.append(n_letters - 1) # EOS
  return torch.LongTensor(letter_indexes)

In [9]:
def randomTrainingExample():
    category, line = randomTrainingPair()
    category_tensor = categoryTensor(category)
    input_line_tensor = inputTensor(line)
    target_line_tensor = targetTensor(line)
    return category_tensor, input_line_tensor, target_line_tensor

In [10]:
category_tensor, input_line_tensor, target_line_tensor = randomTrainingExample()

In [11]:
category_tensor.shape

torch.Size([1, 18])

In [12]:
input_line_tensor[0].shape

torch.Size([1, 59])

In [13]:
target_line_tensor

tensor([ 0,  6,  7,  4, 17, 58])

In [14]:
## Create the network

In [15]:
n_categories

18

In [30]:
class RNN(torch.nn.Module):
  def __init__(self, input_size, hidden_size, output_size):
    super().__init__()

    self.hidden_size = hidden_size
    self.i2o = torch.nn.Linear(n_categories + input_size + hidden_size, output_size)
    self.i2h = torch.nn.Linear(n_categories + input_size + hidden_size, hidden_size)
    self.o2o = torch.nn.Linear(output_size + hidden_size, output_size)
    self.dropout = torch.nn.Dropout(0.1)
    self.softmax = torch.nn.LogSoftmax(dim=1)

  def forward(self, category, input, hidden):
    combined = torch.cat((category, input, hidden), dim=1)
    output = self.i2o(combined)
    hidden = self.i2h(combined)
    out_combined = torch.cat((output, hidden), dim=1)
    output = self.o2o(out_combined)
    output = self.dropout(output)
    output = self.softmax(output)

    return output, hidden

  def initHidden(self):
    return torch.zeros(1, self.hidden_size)


In [61]:
rnn = RNN(n_letters, 128, n_letters)
criterion = torch.nn.NLLLoss()
learning_rate = 0.0005

In [62]:
rnn

RNN(
  (i2o): Linear(in_features=205, out_features=59, bias=True)
  (i2h): Linear(in_features=205, out_features=128, bias=True)
  (o2o): Linear(in_features=187, out_features=59, bias=True)
  (dropout): Dropout(p=0.1, inplace=False)
  (softmax): LogSoftmax(dim=1)
)

In [64]:
def train(category_tensor, input_line_tensor, target_line_tensor):
  hidden = rnn.initHidden()

  rnn.zero_grad()

  loss = 0
  for i in range(input_line_tensor.shape[0]):
    letter_tensor = input_line_tensor[i]
    output, hidden = rnn(category_tensor, letter_tensor, hidden)
    loss += criterion(output[0], target_line_tensor[i])

  loss.backward()

  for p in rnn.parameters():
    p.data.sub_(p.grad.data * learning_rate)


  return output, loss.item() / input_line_tensor.shape[0]

In [65]:
n_iters = 100000
print_every = 5000
plot_every = 500
all_losses = []
total_loss = 0

In [121]:
for iter in range(1, n_iters+1):
  output, loss = train(*randomTrainingExample())
  total_loss += loss

  if iter % print_every == 0:
    print(iter, iter / n_iters * 100, loss)

5000 5.0 2.7973998387654624
10000 10.0 2.332315444946289
15000 15.0 2.794699192047119
20000 20.0 2.491244316101074
25000 25.0 3.139006996154785
30000 30.0 2.89309447152274
35000 35.0 2.788565226963588
40000 40.0 2.777625401814779
45000 45.0 2.344016604953342
50000 50.0 1.6140694618225098
55000 55.00000000000001 1.9034315745035808
60000 60.0 2.248643398284912
65000 65.0 2.2371089935302733
70000 70.0 1.4809532165527344
75000 75.0 2.26784610748291
80000 80.0 1.246039663042341
85000 85.0 2.677596092224121
90000 90.0 2.0182414054870605
95000 95.0 1.8351972897847493
100000 100.0 2.5032078425089517


In [67]:
## Inference

In [95]:
def sample(category, start_letter="A", num_letters=10):
  with torch.no_grad():
    hidden = rnn.initHidden()
    input_tensor = inputTensor(start_letter)
    category_tensor = categoryTensor(category)

    for i in range(num_letters):
      output, hidden = rnn(category_tensor, input_tensor[0], hidden)
      idx = torch.argmax(output, dim=1).item()
      try:
        new_letter = all_letters[idx]
        start_letter += new_letter
        input_tensor = inputTensor(new_letter)
      except IndexError:
        return f">> {start_letter}"

    return f">>> {start_letter}"

In [120]:
print(sample("English", "A"))

>> Allon


In [123]:
def samples(category, start_letters='ABC'):
    for start_letter in start_letters:
        print(sample(category, start_letter))

In [124]:
samples('Russian', 'RUS')

samples('German', 'GER')

samples('Spanish', 'SPA')

samples('Chinese', 'CHI')

>> Roskov
>> Uaran
>> Shiman
>> Gres
>> Ester
>> Roure
>> Salla
>> Paner
>> Alara
>> Cha
>> Han
>> Iun


In [143]:
sample("Dutch", "G")

'>> Gerner'